# 001 ベースライン実験

## 目的

Breast Cancer Wisconsin データセットを使い、最初の分類ベースラインを作成する。

この Notebook では、モデル精度を高めることよりも、以下を優先する。

- データセットの構造を確認する
- train / test split を行う
- 最小限の分類モデルを訓練する
- 評価指標を出力する
- 結果を自分の言葉で観察する

## 実行結果の要約

テストデータ 114 件に対する結果は次のとおり。

| 指標 | 結果 |
|---|---:|
| accuracy | 0.965 |
| precision（悪性） | 0.975 |
| recall（悪性） | 0.929 |
| F1（悪性） | 0.951 |
| 悪性の見逃し | 3 件 |
| 良性の過検出 | 1 件 |

今回のベースラインは全体の正解率が高い一方で、悪性 42 件のうち 3 件を良性と誤判定した。今後は accuracy の改善よりも、悪性の見逃しを減らせるかを優先して検証する。

なお、この結果は単一の train / test split に対する評価であり、一般化性能や医療上の有効性を示すものではない。

## この Notebook の読み方

Notebook は、上から順番に実行する実験ノートである。

各コードセルは、次のどれかを行っている。

1. ライブラリを読み込む
2. データを読み込む
3. データの中身を確認する
4. 訓練用データとテスト用データに分ける
5. モデルを学習する
6. 予測する
7. 評価指標を見る
8. 観察を書く

最初は、各セルを `Shift + Enter` で 1 つずつ実行する。


## 実験メモ

### データセットを選んだ理由

- scikit-learn に内蔵されており、外部データ取得が不要で再現しやすい
- 二値分類問題であり、初回の評価指標設計に集中しやすい
- 特徴量が数値で揃っており、前処理よりも評価とベースライン構築に集中できる

### 今回の完了条件

- データを読み込める
- 特徴量数、サンプル数、目的変数の分布を確認できる
- Logistic Regression でベースラインを作る
- accuracy, precision, recall, F1 を出力する
- confusion matrix を読み取る
- 観察を日本語で書く


## 1. ライブラリを読み込む

ここでは、今回使う道具を読み込む。

- `numpy`: 数値計算
- `pandas`: 表形式データの操作
- `sklearn.datasets`: 練習用データセット
- `train_test_split`: データ分割
- `LogisticRegression`: 最初の分類モデル
- `metrics`: モデル評価


In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)


## 2. データを読み込む

`load_breast_cancer()` は、乳がん診断に関する二値分類データセットを読み込む。

このデータセットでは、目的変数 `target` は次の意味を持つ。

- `0`: malignant（悪性）
- `1`: benign（良性）

ここで注意すること:

機械学習の分類問題では、単に「正解率が高い」だけでは不十分である。
特にこのデータセットでは、**悪性を良性と誤判定する見逃し**が重要な誤りになる。


In [2]:
dataset = load_breast_cancer()

X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name="target")

print("特徴量 X の形状:", X.shape)
print("目的変数 y の形状:", y.shape)
print("目的変数のクラス名:", dataset.target_names)
print()
print("目的変数の意味:")
for class_id, class_name in enumerate(dataset.target_names):
    print(f"  {class_id}: {class_name}")
print()
print("目的変数の分布:")
print(y.value_counts().sort_index())


特徴量 X の形状: (569, 30)
目的変数 y の形状: (569,)
目的変数のクラス名: ['malignant' 'benign']

目的変数の意味:
  0: malignant
  1: benign

目的変数の分布:
target
0    212
1    357
Name: count, dtype: int64


### この出力の読み方

- `特徴量 X の形状` は `(サンプル数, 特徴量数)` を表す
- `目的変数 y の形状` は、正解ラベルの数を表す
- `目的変数の分布` は、悪性と良性がそれぞれ何件あるかを表す

この時点で見るべきこと:

- サンプル数はいくつか
- 特徴量はいくつあるか
- 悪性と良性の件数に偏りがあるか


## 3. 特徴量の先頭を見る

`X.head()` は、特徴量テーブルの先頭5行を表示する。

ここでは、各行が1つのサンプル、各列が1つの特徴量であることを確認する。


In [3]:
X.head()


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 4. 訓練用データとテスト用データに分ける

モデルの性能を見るには、学習に使ったデータだけで評価してはいけない。

そのため、データを次の2つに分ける。

- 訓練用データ: モデルに学習させる
- テスト用データ: 学習後に性能を確認する

`stratify=y` は、悪性と良性の比率が訓練用・テスト用で大きく崩れないようにする指定である。


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("訓練用データ:", X_train.shape, y_train.shape)
print("テスト用データ:", X_test.shape, y_test.shape)
print()
print("訓練用データの目的変数分布:")
print(y_train.value_counts().sort_index())
print()
print("テスト用データの目的変数分布:")
print(y_test.value_counts().sort_index())


訓練用データ: (455, 30) (455,)
テスト用データ: (114, 30) (114,)

訓練用データの目的変数分布:
target
0    170
1    285
Name: count, dtype: int64

テスト用データの目的変数分布:
target
0    42
1    72
Name: count, dtype: int64


## 5. Logistic Regressionで学習する

ここで初めてモデルを作る。

Logistic Regression は、入力された特徴量から、各サンプルが `0: malignant` か `1: benign` かを予測する分類モデルである。

今回の目的は高精度化ではなく、まず「ベースライン」を作ること。
ベースラインとは、今後の改善実験の比較対象になる最初の基準である。


In [5]:
model = LogisticRegression(max_iter=10_000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("予測が完了しました")
print("予測ラベルの分布:")
print(pd.Series(y_pred, name="pred").value_counts().sort_index())


予測が完了しました


予測ラベルの分布:
pred
0    40
1    74
Name: count, dtype: int64


## 6. 評価指標を見る

ここが観察結果を書くための中心である。

今回見る指標:

- `accuracy`: 全体の正解率
- `precision`: そのクラスだと予測したもののうち、実際に正しかった割合
- `recall`: 実際にそのクラスだったものを、どれだけ拾えたか
- `F1`: precision と recall のバランス

重要:

scikit-learn の `precision_score`, `recall_score`, `f1_score` は、指定しない場合 `1` を陽性クラスとして扱う。
このデータセットでは `1 = benign（良性）` である。
しかし医療的に重要なのは、多くの場合 `0 = malignant（悪性）` を見逃さないことである。

そのため、このNotebookでは悪性と良性の両方について指標を出す。


In [6]:
accuracy = accuracy_score(y_test, y_pred)

metrics = pd.DataFrame(
    {
        "score": [
            accuracy,
            precision_score(y_test, y_pred, pos_label=0),
            recall_score(y_test, y_pred, pos_label=0),
            f1_score(y_test, y_pred, pos_label=0),
            precision_score(y_test, y_pred, pos_label=1),
            recall_score(y_test, y_pred, pos_label=1),
            f1_score(y_test, y_pred, pos_label=1),
        ]
    },
    index=[
        "accuracy（全体の正解率）",
        "precision: malignant（悪性と予測したものの正確さ）",
        "recall: malignant（実際の悪性を拾えた割合）",
        "F1: malignant（悪性のprecision/recallのバランス）",
        "precision: benign（良性と予測したものの正確さ）",
        "recall: benign（実際の良性を拾えた割合）",
        "F1: benign（良性のprecision/recallのバランス）",
    ],
)

metrics


,score
accuracy（全体の正解率）,0.964912
precision: malignant（悪性と予測したものの正確さ）,0.975000
recall: malignant（実際の悪性を拾えた割合）,0.928571
F1: malignant（悪性のprecision/recallのバランス）,0.951220
precision: benign（良性と予測したものの正確さ）,0.959459
recall: benign（実際の良性を拾えた割合）,0.986111
F1: benign（良性のprecision/recallのバランス）,0.972603


### 観察欄との対応

Notebook末尾の観察欄には、上の表から次の値を書く。

- `accuracy`: `accuracy（全体の正解率）`
- `precision`: `precision: malignant（悪性と予測したものの正確さ）`
- `recall`: `recall: malignant（実際の悪性を拾えた割合）`
- `F1`: `F1: malignant（悪性のprecision/recallのバランス）`

今回は、悪性検出を重視して `malignant` 側の precision / recall / F1 を記録する。


## 7. confusion matrixを見る

confusion matrix は、予測がどのように当たったか・外れたかを表す表である。

このNotebookでは、行が「実際のラベル」、列が「予測ラベル」である。

|  | 予測: 悪性 | 予測: 良性 |
|---|---:|---:|
| 実際: 悪性 | 悪性を正しく悪性と予測 | 悪性を良性と誤判定 |
| 実際: 良性 | 良性を悪性と誤判定 | 良性を正しく良性と予測 |

特に見るべきなのは、**実際は悪性なのに、良性と予測した数**である。
これは悪性の見逃しであり、医療的には重要な誤りになりやすい。


In [7]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

cm_df = pd.DataFrame(
    cm,
    index=["実際: malignant（悪性）", "実際: benign（良性）"],
    columns=["予測: malignant（悪性）", "予測: benign（良性）"],
)

cm_df


,予測: malignant（悪性）,予測: benign（良性）
実際: malignant（悪性）,39,3
実際: benign（良性）,1,71


In [8]:
true_malignant_pred_malignant = cm_df.loc["実際: malignant（悪性）", "予測: malignant（悪性）"]
false_negative_malignant = cm_df.loc["実際: malignant（悪性）", "予測: benign（良性）"]
false_positive_malignant = cm_df.loc["実際: benign（良性）", "予測: malignant（悪性）"]
true_benign_pred_benign = cm_df.loc["実際: benign（良性）", "予測: benign（良性）"]

print("confusion matrix の読み取り:")
print(f"- 実際に悪性で、悪性と予測できた数: {true_malignant_pred_malignant}")
print(f"- 実際に悪性だが、良性と誤判定した数（見逃し）: {false_negative_malignant}")
print(f"- 実際に良性だが、悪性と誤判定した数（過検出）: {false_positive_malignant}")
print(f"- 実際に良性で、良性と予測できた数: {true_benign_pred_benign}")


confusion matrix の読み取り:
- 実際に悪性で、悪性と予測できた数: 39
- 実際に悪性だが、良性と誤判定した数（見逃し）: 3
- 実際に良性だが、悪性と誤判定した数（過検出）: 1
- 実際に良性で、良性と予測できた数: 71


## 8. classification reportを見る

`classification_report` は、クラスごとの precision / recall / F1 をまとめて出す。

ここでは、上で計算した評価指標と同じ内容が、scikit-learn標準形式で表示される。

最初は読み飛ばしてもよい。
今は、`malignant` 行と `benign` 行が分かれていることだけ確認する。


In [9]:
print(classification_report(y_test, y_pred, target_names=dataset.target_names))


              precision    recall  f1-score   support

   malignant       0.97      0.93      0.95        42
      benign       0.96      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



## 9. 観察を書くための下書きを出力する

次のセルは、観察欄に転記するための下書きを出力する。

数値そのものよりも、最後の「confusion matrix から分かること」を自分の言葉で書くことが重要である。


In [10]:
observation_draft = f"""
- accuracy: {accuracy:.3f}
- precision（悪性）: {precision_score(y_test, y_pred, pos_label=0):.3f}
- recall（悪性）: {recall_score(y_test, y_pred, pos_label=0):.3f}
- F1（悪性）: {f1_score(y_test, y_pred, pos_label=0):.3f}
- confusion matrix から分かること:
  - 実際に悪性だったサンプルのうち、{false_negative_malignant}件を良性と誤判定した。
  - 実際に良性だったサンプルのうち、{false_positive_malignant}件を悪性と誤判定した。
  - このデータセットでは、悪性の見逃しを減らすことが特に重要だと考える。
"""

print(observation_draft)



- accuracy: 0.965
- precision（悪性）: 0.975
- recall（悪性）: 0.929
- F1（悪性）: 0.951
- confusion matrix から分かること:
  - 実際に悪性だったサンプルのうち、3件を良性と誤判定した。
  - 実際に良性だったサンプルのうち、1件を悪性と誤判定した。
  - このデータセットでは、悪性の見逃しを減らすことが特に重要だと考える。



## 観察

- accuracy は `0.965` で、テストデータ 114 件のうち 110 件を正しく分類した。
- 悪性の precision は `0.975` で、悪性と予測した 40 件のうち 39 件が実際に悪性だった。
- 悪性の recall は `0.929` で、実際に悪性だった 42 件のうち 39 件を検出できた。
- 悪性の F1 は `0.951` だった。
- confusion matrix では、悪性の見逃しが 3 件、良性の過検出が 1 件だった。
- accuracy だけを見ると高性能に見えるが、悪性の見逃しが残っているため、今回の用途では recall も併せて評価する必要がある。
- 今回は単一のデータ分割で評価しているため、この数値だけでモデルの安定性は判断できない。

## 次の改善仮説

1. `StandardScaler` で特徴量を標準化し、Logistic Regression の係数推定を安定させる。
2. 交差検証で悪性 recall の平均とばらつきを計測し、単一分割への依存を確認する。
3. `predict_proba()` の判定閾値を調整し、悪性の見逃しと良性の過検出のトレードオフを比較する。
